In [1]:
import os
import numpy as np
import pandas as pd
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition

In [2]:
CUR_DIR = './'
base_dir = os.path.join(CUR_DIR, 'Current_Law', "OUTPUT")
reform_dir = os.path.join(CUR_DIR, 'TCJA_Ext', "OUTPUT")

base_tpi = safe_read_pickle(os.path.join(base_dir, "TPI", "TPI_vars.pkl"))
base_params = safe_read_pickle(os.path.join(base_dir, "model_params.pkl"))
base_ss = safe_read_pickle(os.path.join(base_dir, "SS", "SS_vars.pkl"))
reform_tpi = safe_read_pickle(os.path.join(reform_dir, "TPI", "TPI_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(reform_dir, "model_params.pkl"))
reform_ss = safe_read_pickle(os.path.join(reform_dir, "SS", "SS_vars.pkl"))

In [3]:
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2025, num_years=10, full_break_out=True)
df

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034,SS
0,IIT: Pct Change due to tax rates,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44
1,IIT: Pct Change due to behavior,1.01,0.99,0.98,0.98,0.97,0.96,0.96,0.96,0.96,0.95,0.97,0.97
2,IIT: Pct Change due to macro,-0.02,-0.03,-0.05,-0.06,-0.08,-0.09,-0.11,-0.13,-0.15,-0.17,-0.09,0.17
3,IIT: Overall Pct Change in taxes,-4.45,-4.48,-4.50,-4.52,-4.54,-4.56,-4.58,-4.61,-4.63,-4.65,-4.55,-4.30
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,0.81,0.83,0.86,0.86,0.86,0.85,0.84,0.82,0.80,0.77,0.83,1.72
6,CIT: Pct Change due to macro,0.42,0.32,0.24,0.18,0.14,0.11,0.08,0.07,0.07,0.07,0.17,-0.92
7,CIT: Overall Pct Change in taxes,1.22,1.15,1.10,1.05,1.00,0.96,0.92,0.89,0.86,0.84,1.00,0.80
8,All: Pct Change due to tax rates,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13
9,All: Pct Change due to behavior,1.00,0.98,0.98,0.97,0.96,0.96,0.95,0.95,0.95,0.94,0.96,1.01


In [4]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([5.038, 5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.26,-0.28,-0.30,-0.31,-0.31,-0.33,-0.34,-0.35,-0.37,-0.38,-3.22
9,Rev Change Due to Behavior,0.05,0.05,0.06,0.06,0.06,0.06,0.06,0.07,0.07,0.07,0.60
10,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.01,-0.05
11,Total Revenue Change,-0.21,-0.22,-0.24,-0.25,-0.26,-0.27,-0.28,-0.30,-0.31,-0.32,-2.67


In [5]:
result_df_static = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_wo_behresp.csv', index_col = 0)
result_df_static

,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,Total
Base,4.29,4.82,5.03,5.23,5.44,5.67,5.90,6.14,6.39,6.64,55.57
Reform,4.29,4.49,4.69,4.88,5.09,5.31,5.53,5.76,6.00,6.24,52.29
Difference,0.00,-0.33,-0.34,-0.35,-0.35,-0.36,-0.37,-0.38,-0.39,-0.40,-3.28


In [6]:
result_df_dynamic = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_w_behresp.csv', index_col = 0)
result_df_dynamic

,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,Total
Base,4.29,4.82,5.03,5.23,5.44,5.67,5.90,6.14,6.39,6.64,55.57
Reform,4.29,4.56,4.76,4.96,5.17,5.39,5.61,5.85,6.09,6.33,53.00
Difference,0.00,-0.26,-0.27,-0.28,-0.27,-0.28,-0.29,-0.30,-0.30,-0.31,-2.57


In [7]:
# Or we can use the Tax-Calc baseline for a direct comparison
base_revenue = result_df_static.loc["Base", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.23,-0.26,-0.27,-0.28,-0.30,-0.31,-0.32,-0.33,-0.35,-0.36,-3.02
1,Rev Change Due to Behavior,0.04,0.05,0.05,0.05,0.05,0.05,0.06,0.06,0.06,0.06,0.54
2,Rev Change Due to Macro,-0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.01,-0.05
3,Total Revenue Change,-0.19,-0.22,-0.23,-0.24,-0.25,-0.26,-0.27,-0.28,-0.30,-0.31,-2.53


In [8]:
# jason's get-around

df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_static.loc["Difference", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = tc_diff * df.loc[1, df_levels.columns[1:]]/df.loc[0, df_levels.columns[1:]]
df_levels.loc[2, df_levels.columns[1:]] = tc_diff * df.loc[2, df_levels.columns[1:]]/df.loc[0, df_levels.columns[1:]]
df_levels.loc[3, df_levels.columns[1:]] = tc_diff * (df.loc[0, df_levels.columns[1:]]+df.loc[1, df_levels.columns[1:]]+df.loc[2, df_levels.columns[1:]])/df.loc[0, df_levels.columns[1:]]
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,0.00,-0.33,-0.34,-0.35,-0.35,-0.36,-0.37,-0.38,-0.39,-0.40,-3.28
1,Rev Change Due to Behavior,-0.00,0.06,0.06,0.06,0.06,0.06,0.07,0.07,0.07,0.07,0.58
2,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.01,-0.06
3,Total Revenue Change,0.00,-0.27,-0.28,-0.29,-0.29,-0.30,-0.31,-0.32,-0.33,-0.34,-2.75


In [9]:
df_levels.to_csv('og_usa_result_w_tcja.csv')

In [10]:
df_levels['2025-2034'][0]+df_levels['2025-2034'][1]

-2.6939693723264737